In [17]:
import sys, os
import numpy as np
import pandas as pd
import pickle
import gc
import matplotlib.pyplot as plt
import pathlib
import lightgbm as lgb
import logging
import warnings
import tensorflow as tf
import scipy.ndimage as ndi
warnings.simplefilter('ignore')
from tqdm.notebook import tqdm
from glob import glob
from sklearn.model_selection import GroupKFold, KFold, StratifiedKFold, StratifiedGroupKFold
from sklearn import cluster
from sklearn.metrics import precision_score, average_precision_score
from sklearn.metrics import precision_score, average_precision_score
from tensorflow import keras
from tensorflow.keras.callbacks import EarlyStopping, LearningRateScheduler, ModelCheckpoint, ReduceLROnPlateau
from tensorflow.keras.optimizers.schedules import ExponentialDecay, CosineDecay

DATA_DIR = "/kaggle/input/tlvmc-parkinsons-freezing-gait-prediction/"
train = False # trains the models if True; makes predictions using trained models if False

In [18]:
# DATA LOADING WITH STRATIFIED GROUP K-FOLD

import pandas as pd
import numpy as np
from glob import glob
from sklearn.model_selection import StratifiedGroupKFold
from tqdm import tqdm

DATA_DIR = "/kaggle/input/tlvmc-parkinsons-freezing-gait-prediction/"
train = False  # Set to False for inference

# Load metadata
subjects = pd.read_csv(f"{DATA_DIR}subjects.csv")
meta_tdcs = pd.read_csv(f"{DATA_DIR}tdcsfog_metadata.csv")
meta_defog = pd.read_csv(f"{DATA_DIR}defog_metadata.csv")

print("=" * 80)
print("IMPLEMENTING STRATIFIED GROUP K-FOLD SPLIT")
print("=" * 80)

# STEP 1: Analyze TDCSFOG data and create stratified splits

if train:
    print("\n--- Processing TDCSFOG data ---")
    
    # Count positive instances for each recording
    tdcs_stats = []
    for f in tqdm(meta_tdcs['Id'], desc="Analyzing TDCSFOG"):
        fpath = f"{DATA_DIR}train/tdcsfog/{f}.csv"
        if not os.path.exists(fpath):
            continue
            
        df = pd.read_csv(fpath)
        tdcs_stats.append({
            'Id': f,
            'Subject': meta_tdcs[meta_tdcs['Id'] == f]['Subject'].values[0],
            'StartHesitation': np.sum(df['StartHesitation']),
            'Turn': np.sum(df['Turn']),
            'Walking': np.sum(df['Walking']),
            'Total': len(df),
            'Data': 'tdcsfog'
        })
    
    tdcs_df = pd.DataFrame(tdcs_stats)
    print(f"TDCSFOG recordings: {len(tdcs_df)}")
    print(f"  StartHesitation events: {tdcs_df['StartHesitation'].sum():,}")
    print(f"  Turn events: {tdcs_df['Turn'].sum():,}")
    print(f"  Walking events: {tdcs_df['Walking'].sum():,}")
    
    # Create stratified splits for TDCSFOG
    # Use 5-fold and select fold 2 (most balanced as found in PyTorch model)
    sgkf = StratifiedGroupKFold(n_splits=5, random_state=42, shuffle=True)
    
    # We stratify by dummy variable (all 1s) but group by Subject
    # This ensures patients don't appear in both train and validation
    X = tdcs_df['Id'].values
    y = np.ones(len(tdcs_df))  # Dummy labels (stratification happens via groups)
    groups = tdcs_df['Subject'].values
    
    print("\nAnalyzing TDCSFOG folds:")
    for fold_idx, (train_idx, valid_idx) in enumerate(sgkf.split(X, y, groups)):
        print(f"\nFold {fold_idx}:")
        print(f"  Train: {len(train_idx)} recordings")
        print(f"  Valid: {len(valid_idx)} recordings")
        print(f"  Train events: Start={tdcs_df.iloc[train_idx]['StartHesitation'].sum():,}, "
              f"Turn={tdcs_df.iloc[train_idx]['Turn'].sum():,}, "
              f"Walk={tdcs_df.iloc[train_idx]['Walking'].sum():,}")
        print(f"  Valid events: Start={tdcs_df.iloc[valid_idx]['StartHesitation'].sum():,}, "
              f"Turn={tdcs_df.iloc[valid_idx]['Turn'].sum():,}, "
              f"Walk={tdcs_df.iloc[valid_idx]['Walking'].sum():,}")
    
    # Select fold 2 (most balanced)
    SELECTED_FOLD_TDCS = 2
    for fold_idx, (train_idx, valid_idx) in enumerate(sgkf.split(X, y, groups)):
        if fold_idx == SELECTED_FOLD_TDCS:
            tdcs_train_ids = tdcs_df.iloc[train_idx]['Id'].tolist()
            tdcs_valid_ids = tdcs_df.iloc[valid_idx]['Id'].tolist()
            break
    
    print(f"\n*** Selected Fold {SELECTED_FOLD_TDCS} for TDCSFOG ***")
    print(f"Train IDs: {len(tdcs_train_ids)}, Valid IDs: {len(tdcs_valid_ids)}")


    # STEP 2: Analyze DEFOG data and create stratified splits
    
    print("\n--- Processing DEFOG data ---")
    
    defog_stats = []
    for f in tqdm(meta_defog['Id'], desc="Analyzing DEFOG"):
        fpath = f"{DATA_DIR}train/defog/{f}.csv"
        if not os.path.exists(fpath):
            continue
            
        df = pd.read_csv(fpath)
        defog_stats.append({
            'Id': f,
            'Subject': meta_defog[meta_defog['Id'] == f]['Subject'].values[0],
            'StartHesitation': np.sum(df['StartHesitation']),
            'Turn': np.sum(df['Turn']),
            'Walking': np.sum(df['Walking']),
            'Total': len(df),
            'Data': 'defog'
        })
    
    defog_df = pd.DataFrame(defog_stats)
    print(f"DEFOG recordings: {len(defog_df)}")
    print(f"  StartHesitation events: {defog_df['StartHesitation'].sum():,}")
    print(f"  Turn events: {defog_df['Turn'].sum():,}")
    print(f"  Walking events: {defog_df['Walking'].sum():,}")
    
    # Create stratified splits for DEFOG
    sgkf = StratifiedGroupKFold(n_splits=5, random_state=42, shuffle=True)
    X = defog_df['Id'].values
    y = np.ones(len(defog_df))
    groups = defog_df['Subject'].values
    
    print("\nAnalyzing DEFOG folds:")
    for fold_idx, (train_idx, valid_idx) in enumerate(sgkf.split(X, y, groups)):
        print(f"\nFold {fold_idx}:")
        print(f"  Train: {len(train_idx)} recordings")
        print(f"  Valid: {len(valid_idx)} recordings")
        print(f"  Train events: Start={defog_df.iloc[train_idx]['StartHesitation'].sum():,}, "
              f"Turn={defog_df.iloc[train_idx]['Turn'].sum():,}, "
              f"Walk={defog_df.iloc[train_idx]['Walking'].sum():,}")
        print(f"  Valid events: Start={defog_df.iloc[valid_idx]['StartHesitation'].sum():,}, "
              f"Turn={defog_df.iloc[valid_idx]['Turn'].sum():,}, "
              f"Walk={defog_df.iloc[valid_idx]['Walking'].sum():,}")
    
    # Select fold 1 (most balanced for DEFOG as found in PyTorch model)
    SELECTED_FOLD_DEFOG = 1
    for fold_idx, (train_idx, valid_idx) in enumerate(sgkf.split(X, y, groups)):
        if fold_idx == SELECTED_FOLD_DEFOG:
            defog_train_ids = defog_df.iloc[train_idx]['Id'].tolist()
            defog_valid_ids = defog_df.iloc[valid_idx]['Id'].tolist()
            break
    
    print(f"\n*** Selected Fold {SELECTED_FOLD_DEFOG} for DEFOG ***")
    print(f"Train IDs: {len(defog_train_ids)}, Valid IDs: {len(defog_valid_ids)}")
    
    
    # STEP 3: Combine splits and create id_info dataframe
    
    print("\n" + "=" * 80)
    print("FINAL TRAIN/VALIDATION SPLIT")
    print("=" * 80)
    
    # Combine all train and validation IDs
    all_train_ids = tdcs_train_ids + defog_train_ids
    all_valid_ids = tdcs_valid_ids + defog_valid_ids
    
    # Create id_info with split information
    id_info_list = []
    
    # Add TDCSFOG train
    for id_ in tdcs_train_ids:
        id_info_list.append({
            'Id': id_,
            'Subject': tdcs_df[tdcs_df['Id'] == id_]['Subject'].values[0],
            'Data': 'tdcsfog',
            'Split': 'train'
        })
    
    # Add TDCSFOG valid
    for id_ in tdcs_valid_ids:
        id_info_list.append({
            'Id': id_,
            'Subject': tdcs_df[tdcs_df['Id'] == id_]['Subject'].values[0],
            'Data': 'tdcsfog',
            'Split': 'valid'
        })
    
    # Add DEFOG train
    for id_ in defog_train_ids:
        id_info_list.append({
            'Id': id_,
            'Subject': defog_df[defog_df['Id'] == id_]['Subject'].values[0],
            'Data': 'defog',
            'Split': 'train'
        })
    
    # Add DEFOG valid
    for id_ in defog_valid_ids:
        id_info_list.append({
            'Id': id_,
            'Subject': defog_df[defog_df['Id'] == id_]['Subject'].values[0],
            'Data': 'defog',
            'Split': 'valid'
        })
    
    id_info = pd.DataFrame(id_info_list)
    
    # Calculate final statistics
    train_tdcs_stats = tdcs_df[tdcs_df['Id'].isin(tdcs_train_ids)]
    train_defog_stats = defog_df[defog_df['Id'].isin(defog_train_ids)]
    valid_tdcs_stats = tdcs_df[tdcs_df['Id'].isin(tdcs_valid_ids)]
    valid_defog_stats = defog_df[defog_df['Id'].isin(defog_valid_ids)]
    
    print("\nTRAINING SET:")
    print(f"  Total recordings: {len(all_train_ids)}")
    print(f"    TDCSFOG: {len(tdcs_train_ids)}")
    print(f"    DEFOG: {len(defog_train_ids)}")
    print(f"  FOG Events:")
    print(f"    StartHesitation: {train_tdcs_stats['StartHesitation'].sum() + train_defog_stats['StartHesitation'].sum():,}")
    print(f"    Turn: {train_tdcs_stats['Turn'].sum() + train_defog_stats['Turn'].sum():,}")
    print(f"    Walking: {train_tdcs_stats['Walking'].sum() + train_defog_stats['Walking'].sum():,}")
    
    print("\nVALIDATION SET:")
    print(f"  Total recordings: {len(all_valid_ids)}")
    print(f"    TDCSFOG: {len(tdcs_valid_ids)}")
    print(f"    DEFOG: {len(defog_valid_ids)}")
    print(f"  FOG Events:")
    print(f"    StartHesitation: {valid_tdcs_stats['StartHesitation'].sum() + valid_defog_stats['StartHesitation'].sum():,}")
    print(f"    Turn: {valid_tdcs_stats['Turn'].sum() + valid_defog_stats['Turn'].sum():,}")
    print(f"    Walking: {valid_tdcs_stats['Walking'].sum() + valid_defog_stats['Walking'].sum():,}")
    
    print("\n" + "=" * 80)
    print("Stratified split complete! All FOG event types in validation ✓")
    print("=" * 80)

else:
    # For inference mode (train=False), load test data
    id_tdcs = set([os.path.basename(f).split('.')[0] for f in glob(f"{DATA_DIR}test/tdcsfog/*.csv")])
    id_defog = set([os.path.basename(f).split('.')[0] for f in glob(f"{DATA_DIR}test/defog/*.csv")])
    
    ids = []
    subs = []
    datanames = []
    
    for si in subjects['Subject'].values:
        idi = meta_tdcs[meta_tdcs['Subject']==si]['Id'].tolist() + meta_defog[meta_defog['Subject']==si]['Id'].tolist()
        for i in idi:
            if i in id_defog:
                ids.append(i)
                subs.append(si)
                datanames.append('defog')
            elif i in id_tdcs:
                ids.append(i)
                subs.append(si)
                datanames.append('tdcsfog')
    
    id_info = pd.DataFrame({'Id': ids, 'Subject': subs, 'Data': datanames, 'Split': 'test'})
    id_info.drop_duplicates(inplace=True)

IMPLEMENTING STRATIFIED GROUP K-FOLD SPLIT


## Common func

In [19]:
def precision(preds, gts):
    metrics = []
    metrics.append(average_precision_score(gts[:, 0].flatten()>0, preds[:, 0].flatten()))
    metrics.append(average_precision_score(gts[:, 1].flatten()>0, preds[:, 1].flatten()))
    metrics.append(average_precision_score(gts[:, 2].flatten()>0, preds[:, 2].flatten()))

    print(metrics)
    print(np.mean(metrics))
    return np.mean(metrics)

In [20]:
def resize_func(x, target_size, use_percentile_feat=False):
    ch = x.shape[0] 
    input_size = x.shape[1]
    
    pad = target_size - input_size % target_size
    factor = (input_size + pad) / input_size

    x = np.array([ndi.zoom(xi, zoom=factor, mode='reflect') for xi in x])
    x = x.reshape((ch, target_size, -1))

    res = {} 
    res['mean'] = np.mean(x, axis=2).reshape(ch, -1)
    res['max'] = np.max(x, axis=2).reshape(ch, -1)
    res['min'] = np.min(x, axis=2).reshape(ch, -1)
    res['med'] = np.median(x, axis=2).reshape(ch, -1)
    res['std'] = np.sqrt(np.var(x, axis=2).reshape(ch, -1))
    if use_percentile_feat:
        for p in [15, 30, 45, 60, 75, 90]:
            res[f"p{p}"] = np.percentile(x, [p], axis=2).reshape(ch, -1)

    return res

def resize_sequence(filename, target_size):
    f = pd.read_csv(filename)
    data = filename.split('/')[-2]
    
    x_cols = ['AccV', 'AccML', 'AccAP']
    y_cols = ['Event'] if data == 'notype' else ['StartHesitation', 'Turn', 'Walking']
    p_cols = ['p15', 'p30', 'p45', 'p60', 'p75', 'p90']
    res_cols = ['mean', 'max', 'min', 'med', 'std']
    
    res_x = resize_func(f[x_cols].values.transpose(), target_size, CFG.use_percentile_feat)
    if CFG.train:
        res_y = resize_func(f[y_cols].values.transpose(), target_size, False)
        
    res = {}
    for k in res_cols:
        res[f"{x_cols[0]}{k}"] = res_x[k][0]
        res[f"{x_cols[1]}{k}"] = res_x[k][1]
        res[f"{x_cols[2]}{k}"] = res_x[k][2]
        
        if CFG.train:
            # there are just dummy. validation loss calculated by those values are not monitored. 
            if data == 'notype':
                res[f"StartHesitation{k}"] = np.zeros_like(res_y[k][0]).astype(res_y[k][0].dtype)
                res[f"Turn{k}"] = res_y[k][0]
                res[f"Walking{k}"] = np.zeros_like(res_y[k][0]).astype(res_y[k][0].dtype)
            else:
                res[f"{y_cols[0]}{k}"] = res_y[k][0]
                res[f"{y_cols[1]}{k}"] = res_y[k][1]
                res[f"{y_cols[2]}{k}"] = res_y[k][2]
        
        if CFG.use_percentile_feat: 
            for k in p_cols:
                res[f"{x_cols[0]}{k}"] = res_x[k][0]
                res[f"{x_cols[1]}{k}"] = res_x[k][1]
                res[f"{x_cols[2]}{k}"] = res_x[k][2]
        
    
    res = pd.DataFrame(res)
    return res

def save_resized_data(data, target_size):
    out = f"resized_sequences/{data}/"
    os.makedirs(out, exist_ok=True)
    files =  glob(f"{DATA_DIR}/train/{data}/*.csv") if CFG.train else glob(f"{DATA_DIR}/test/{data}/*.csv")
    for fi in tqdm(files):
        name = fi.split('/')[-1]
        res = resize_sequence(fi, target_size)
        res.to_csv(f"{out}{name}", index=False)

In [21]:
def get_data(valid=False):
    """
    Load training or validation data based on stratified split
    
    Args:
        valid: If True, load validation data. If False, load training data.
    
    Returns:
        If CFG.train=True: (x_data, y_data, names)
        If CFG.train=False: (x_data, names)
    """
    use_cols = []
    cols = ['mean', 'max', 'min', 'med', 'std']
    if CFG.use_percentile_feat:
        cols += ['p15', 'p30', 'p45', 'p60', 'p75', 'p90'] 
    for c in cols:
        use_cols += [f"AccV{c}", f"AccML{c}", f"AccAP{c}"]
    
    # NEW: Use 'Split' column instead of 'Data' type
    if CFG.train:
        # Training mode: use stratified split
        bools = id_info['Split'] == 'valid' if valid else id_info['Split'] == 'train'
    else:
        # Inference mode: use all test data
        bools = id_info['Split'] == 'test'
    
    x_data, y_data, names = [], [], []
    
    for _, row in tqdm(id_info[bools].iterrows(), total=len(id_info[bools]), 
                      desc=f"Loading {'validation' if valid else 'training'} data"):
        di = row['Data']
        idi = row['Id']
        
        if di == '':
            continue
            
        fpath = f"resized_sequences/{di}/{idi}.csv"
        if not os.path.exists(fpath):
            continue
            
        f = pd.read_csv(fpath)
        f.ffill(inplace=True)
        
        x = f[use_cols].values.astype(np.float32).reshape(-1, CFG.target_size, len(use_cols))
        names.append(f"{di}/{idi}.csv")
        x_data.append(x)
        
        if CFG.train:
            y = f[['StartHesitationmax', 'Turnmax', 'Walkingmax']].values.astype(np.float32).reshape(-1, CFG.target_size, 3)
            y_data.append(y)
    
    if len(x_data) == 0:
        raise ValueError(f"No data found for {'validation' if valid else 'training'} set!")
    
    x_data = np.concatenate(x_data)
    
    if CFG.train:
        y_data = np.concatenate(y_data)
        
        # Add NoEvent class (complement of all events)
        noevent = np.expand_dims(np.sum(y_data, axis=2) < 1, axis=2).astype('float32')
        y_data = np.concatenate([y_data, noevent], axis=2)
        
        # Normalize overlapping events
        y_data[np.sum(y_data, axis=2) == 2] /= 2
        y_data[np.sum(y_data, axis=2) == 3] /= 3
        
        return x_data, y_data, names
    
    return x_data, names



In [22]:
# Evaluate only presence of Event.
def loss_func(y_true, preds):
    return tf.reduce_mean(keras.losses.binary_crossentropy(1.0 - y_true[:, :, -1], 1.0 - preds[:, :, -1]))

## Model

In [23]:
def load_bidLSTM(seq_len, n_feat, hidden0, hidden1, hidden2, hidden3, dense, n_class=4):
    model = keras.models.Sequential([
            keras.layers.Input(shape = (seq_len, n_feat)),
            keras.layers.Bidirectional(keras.layers.LSTM(hidden0, return_sequences = True)),
            keras.layers.Bidirectional(keras.layers.LSTM(hidden1, return_sequences = True)),
            keras.layers.Bidirectional(keras.layers.LSTM(hidden2, return_sequences = True)),
            keras.layers.Bidirectional(keras.layers.LSTM(hidden3, return_sequences = True)),
            keras.layers.Dense(dense, activation = 'selu'),
            keras.layers.Dense(n_class, activation='softmax'),
        ])
    return model

In [24]:
# Implementation of https://www.kaggle.com/competitions/ventilator-pressure-prediction/discussion/285330
def load_1dCNN(seq_len, n_feat, n_class=4):
    inputs_x = keras.layers.Input(shape=(seq_len, n_feat))

    c0 = keras.layers.Conv1D(n_feat, 2, padding='same')(inputs_x)
    c1 = keras.layers.Conv1D(n_feat, 3, padding='same')(inputs_x)
    c2 = keras.layers.Conv1D(n_feat, 4, padding='same')(inputs_x)

    x = tf.keras.layers.Concatenate(axis=2)([inputs_x, c0, c1, c2])
    x = keras.layers.Bidirectional(keras.layers.LSTM(1024, return_sequences=True, dropout=0.1))(x)
    x = keras.layers.Bidirectional(keras.layers.LSTM(512, return_sequences=True, dropout=0.1))(x)
    x = keras.layers.Bidirectional(keras.layers.LSTM(256, return_sequences=True, dropout=0.1))(x)
    x = keras.layers.Bidirectional(keras.layers.LSTM(128, return_sequences=True, dropout=0.1))(x)

    x = tf.keras.layers.Dense(n_class, activation="softmax")(x)
    model = tf.keras.Model(inputs=inputs_x, outputs=x)

    return model

## Training
All labeled data are used as training data. Validation score is calucrated by only 'NoEvent' data. 

In [25]:
# 1dCNN (with percentile features)
class CFG:
    exp = '001'
    target_size = 2048
    lr = 3e-4
    early_stop_patience = 30
    epoch = 30
    batch_size = 16
    model_no = 2
    use_percentile_feat = True
    train = train
    
if CFG.train:
    save_resized_data('tdcsfog', CFG.target_size)
    save_resized_data('defog', CFG.target_size)
    save_resized_data('notype', CFG.target_size)
    
    gpu_strategy = tf.distribute.get_strategy()
    with gpu_strategy.scope():
        valid_x, valid_y, _ = get_data(True)
        train_x, train_y, _ = get_data(False)

        model = load_bidLSTM(train_x.shape[1], train_x.shape[2], 1024, 512, 256, 128, 128)
        scheduler = CosineDecay(CFG.lr, CFG.epoch * (train_x.shape[0] // CFG.batch_size))
        optimizer = keras.optimizers.Adam(learning_rate = scheduler)
        model.compile(optimizer=optimizer, loss='categorical_crossentropy', metrics=[loss_func, 'accuracy'])

        callbacks = []
        callbacks.append(
            EarlyStopping(
                monitor = "val_loss_func",
                patience = CFG.early_stop_patience,
                verbose =1,
                mode = "min",
                restore_best_weights = True))

        history = model.fit(train_x, train_y, validation_data=(valid_x, valid_y), epochs=CFG.epoch, batch_size=CFG.batch_size, callbacks=callbacks) 
        save_locally = tf.saved_model.SaveOptions(experimental_io_device='/job:localhost')
        model.save(f'model-{CFG.model_no}', options=save_locally)

        preds = 1.0 - model.predict(valid_x, batch_size=CFG.batch_size, verbose=2)[:, :, -1]
        # It is not a full size score. If you want to see the actual score, you need to resize it.
        print(average_precision_score((1.0 - valid_y[:, :, -1]).flatten() > 0, preds.flatten()))

In [26]:
from sklearn.metrics import average_precision_score
import numpy as np
import matplotlib.pyplot as plt

if train:
    print("=" * 80)
    print("MODEL EVALUATION - Average Precision Scores")
    print("=" * 80)
    
    # Get predictions on validation set
    print("\nGenerating predictions on validation set...")
    valid_preds = model.predict(valid_x, batch_size=CFG.batch_size, verbose=1)
    
    # Extract predictions for each class
    # valid_preds shape: (num_samples, 2048, 4) -> [StartHesitation, Turn, Walking, NoEvent]
    # valid_y shape: (num_samples, 2048, 4)
    
    # Flatten predictions and ground truth for evaluation
    pred_start = valid_preds[:, :, 0].flatten()  # StartHesitation predictions
    pred_turn = valid_preds[:, :, 1].flatten()   # Turn predictions  
    pred_walk = valid_preds[:, :, 2].flatten()   # Walking predictions
    
    true_start = valid_y[:, :, 0].flatten()      # StartHesitation ground truth
    true_turn = valid_y[:, :, 1].flatten()       # Turn ground truth
    true_walk = valid_y[:, :, 2].flatten()       # Walking ground truth
    
    # Convert to binary (is event present?)
    true_start_binary = (true_start > 0).astype(int)
    true_turn_binary = (true_turn > 0).astype(int)
    true_walk_binary = (true_walk > 0).astype(int)
    
    # Compute Average Precision for each class
    print("\n" + "-" * 80)
    print("AVERAGE PRECISION SCORES (per class):")
    print("-" * 80)
    
    ap_start = average_precision_score(true_start_binary, pred_start)
    ap_turn = average_precision_score(true_turn_binary, pred_turn)
    ap_walk = average_precision_score(true_walk_binary, pred_walk)
    
    print(f"StartHesitation AP: {ap_start:.4f}")
    print(f"Turn AP:            {ap_turn:.4f}")
    print(f"Walking AP:         {ap_walk:.4f}")
    
    # Overall Average Precision (mean of the three classes)
    overall_ap = np.mean([ap_start, ap_turn, ap_walk])
    print(f"\nOVERALL AP (Mean):  {overall_ap:.4f}")
    print("=" * 80)
    
    # Additional metrics: Event detection (ignoring class)
    any_event_pred = 1.0 - valid_preds[:, :, 3].flatten()  # 1 - NoEvent = Any event
    any_event_true = 1.0 - valid_y[:, :, 3].flatten()
    any_event_binary = (any_event_true > 0).astype(int)
    
    ap_event_detection = average_precision_score(any_event_binary, any_event_pred)
    print(f"\nEvent Detection AP (any FOG): {ap_event_detection:.4f}")
    print("(This is the score used for early stopping via loss_func)")
    
    # Class distribution in validation set
    print("\n" + "-" * 80)
    print("VALIDATION SET STATISTICS:")
    print("-" * 80)
    total_timesteps = len(true_start_binary)
    n_start = true_start_binary.sum()
    n_turn = true_turn_binary.sum()
    n_walk = true_walk_binary.sum()
    n_noevent = total_timesteps - n_start - n_turn - n_walk
    
    print(f"Total timesteps:     {total_timesteps:,}")
    print(f"StartHesitation:     {n_start:,} ({100*n_start/total_timesteps:.2f}%)")
    print(f"Turn:                {n_turn:,} ({100*n_turn/total_timesteps:.2f}%)")
    print(f"Walking:             {n_walk:,} ({100*n_walk/total_timesteps:.2f}%)")
    print(f"NoEvent:             {n_noevent:,} ({100*n_noevent/total_timesteps:.2f}%)")
    
    # Precision-Recall curves
    print("\n" + "-" * 80)
    print("GENERATING PRECISION-RECALL CURVES...")
    print("-" * 80)
    
    from sklearn.metrics import precision_recall_curve
    
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    
    # StartHesitation
    precision_start, recall_start, _ = precision_recall_curve(true_start_binary, pred_start)
    axes[0, 0].plot(recall_start, precision_start, linewidth=2)
    axes[0, 0].set_xlabel('Recall', fontsize=12)
    axes[0, 0].set_ylabel('Precision', fontsize=12)
    axes[0, 0].set_title(f'StartHesitation (AP = {ap_start:.4f})', fontsize=14, fontweight='bold')
    axes[0, 0].grid(True, alpha=0.3)
    axes[0, 0].set_xlim([0, 1])
    axes[0, 0].set_ylim([0, 1])
    
    # Turn
    precision_turn, recall_turn, _ = precision_recall_curve(true_turn_binary, pred_turn)
    axes[0, 1].plot(recall_turn, precision_turn, linewidth=2, color='orange')
    axes[0, 1].set_xlabel('Recall', fontsize=12)
    axes[0, 1].set_ylabel('Precision', fontsize=12)
    axes[0, 1].set_title(f'Turn (AP = {ap_turn:.4f})', fontsize=14, fontweight='bold')
    axes[0, 1].grid(True, alpha=0.3)
    axes[0, 1].set_xlim([0, 1])
    axes[0, 1].set_ylim([0, 1])
    
    # Walking
    precision_walk, recall_walk, _ = precision_recall_curve(true_walk_binary, pred_walk)
    axes[1, 0].plot(recall_walk, precision_walk, linewidth=2, color='green')
    axes[1, 0].set_xlabel('Recall', fontsize=12)
    axes[1, 0].set_ylabel('Precision', fontsize=12)
    axes[1, 0].set_title(f'Walking (AP = {ap_walk:.4f})', fontsize=14, fontweight='bold')
    axes[1, 0].grid(True, alpha=0.3)
    axes[1, 0].set_xlim([0, 1])
    axes[1, 0].set_ylim([0, 1])
    
    # Overall (any event)
    precision_event, recall_event, _ = precision_recall_curve(any_event_binary, any_event_pred)
    axes[1, 1].plot(recall_event, precision_event, linewidth=2, color='red')
    axes[1, 1].set_xlabel('Recall', fontsize=12)
    axes[1, 1].set_ylabel('Precision', fontsize=12)
    axes[1, 1].set_title(f'Any Event Detection (AP = {ap_event_detection:.4f})', fontsize=14, fontweight='bold')
    axes[1, 1].grid(True, alpha=0.3)
    axes[1, 1].set_xlim([0, 1])
    axes[1, 1].set_ylim([0, 1])
    
    plt.suptitle(f'Precision-Recall Curves - Overall AP: {overall_ap:.4f}', 
                 fontsize=16, fontweight='bold', y=1.00)
    plt.tight_layout()
    plt.show()
    
    print("\nEvaluation complete!")
    print("=" * 80)
    
    
    # OPTIONAL: Save scores to file for later reference
    
    scores = {
        'model_no': CFG.model_no,
        'target_size': CFG.target_size,
        'use_percentile_feat': CFG.use_percentile_feat,
        'epochs_trained': len(history.history['loss']),
        'ap_start_hesitation': ap_start,
        'ap_turn': ap_turn,
        'ap_walking': ap_walk,
        'ap_overall': overall_ap,
        'ap_event_detection': ap_event_detection,
    }
    
    print("\nModel Performance Summary:")
    print(f"Model: {CFG.model_no}")
    print(f"Sequence Length: {CFG.target_size}")
    print(f"Percentile Features: {CFG.use_percentile_feat}")
    print(f"Epochs Trained: {scores['epochs_trained']}")
    print(f"Overall AP: {overall_ap:.4f}")
    
    # Save to JSON
    import json
    with open(f'model_{CFG.model_no}_scores.json', 'w') as f:
        json.dump(scores, f, indent=2)
    print(f"\nScores saved to: model_{CFG.model_no}_scores.json")

In [27]:
import matplotlib.pyplot as plt

if train:
    # Plotting Training & Validation Loss and Accuracy
    plt.figure(figsize=(15, 5))
    
    # Loss plot
    plt.subplot(1, 3, 1)
    plt.plot(history.history['loss'], label='Training Loss')
    plt.plot(history.history['val_loss'], label='Validation Loss')
    plt.title('Model Loss Over Epochs')
    plt.xlabel('Epochs')
    plt.ylabel('Loss')
    plt.legend()
    plt.grid(True, alpha=0.3)
    
    # Accuracy plot
    plt.subplot(1, 3, 2)
    plt.plot(history.history['accuracy'], label='Training Accuracy')
    plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
    plt.title('Model Accuracy Over Epochs')
    plt.xlabel('Epochs')
    plt.ylabel('Accuracy')
    plt.legend()
    plt.grid(True, alpha=0.3)
    
    # Custom Metric (loss_func) plot
    plt.subplot(1, 3, 3)
    plt.plot(history.history['loss_func'], label='Training loss_func')
    plt.plot(history.history['val_loss_func'], label='Validation loss_func')
    plt.title('Custom Metric (loss_func) Over Epochs')
    plt.xlabel('Epochs')
    plt.ylabel('Score')
    plt.legend()
    plt.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

## Inference

In [28]:
preds= []
expid = '035'

In [29]:
if not train:  # Only run during inference, not training
    # 1dCNN (percentile features are added)
    class CFG:
        exp = expid
        target_size = 2048
        use_percentile_feat = True
        model_no = 5
        batch_size = 32
        train = False  # ✅ FIXED
        
    save_resized_data('tdcsfog', CFG.target_size)
    save_resized_data('defog', CFG.target_size)
    test_x, names = get_data(False)  # ✅ FIXED

    gpu_strategy = tf.distribute.get_strategy()
    with gpu_strategy.scope():
        model = keras.models.load_model(f"/kaggle/input/pfgp-exp{CFG.exp}/model-{CFG.model_no}", custom_objects={'loss_func':loss_func})
        preds.append(model.predict(test_x, batch_size=CFG.batch_size, verbose=2))
            
    del test_x
    gc.collect()

Loading training data: 100%|██████████| 2/2 [00:00<00:00, 60.42it/s]


1/1 - 5s - 5s/epoch - 5s/step


## Ensemble

In [30]:
if not train:  # Only run during inference, not training
    tcols = ['StartHesitation', 'Turn', 'Walking']
    scols = ['Id', 'StartHesitation', 'Turn' , 'Walking']

    def reconst_sequence(x, target_size):
        res = np.array([
            ndi.zoom(x[:, 0], zoom=target_size/len(x), mode='nearest'),
            ndi.zoom(x[:, 1], zoom=target_size/len(x), mode='nearest'),
            ndi.zoom(x[:, 2], zoom=target_size/len(x), mode='nearest')])
        return res

    dfs = []
    for p in tqdm(preds):
        dfi = []
        for pi, ni in zip(p, names):
            ori = pd.read_csv(f"{DATA_DIR}test/{ni}")

            res = reconst_sequence(pi, len(ori))
            df = pd.DataFrame(res.transpose(), columns=tcols)
            df['Id'] = ni.split('/')[-1].replace('.csv', '') + '_' + df.index.astype(str)
            dfi.append(df[scols])
            
        dfs.append(pd.concat(dfi))

    for c in tcols:
        dfs[0][c] = np.mean([dfs[i][c].values for i in range(len(preds))], axis=0)

100%|██████████| 1/1 [00:00<00:00,  2.01it/s]


In [31]:
if not train:  # Only run during inference, not training
    sub = pd.read_csv(f'{DATA_DIR}sample_submission.csv')
    sub['t'] = 0
    res = pd.merge(sub[['Id','t']], dfs[0], how='left', on='Id').fillna(0.0)
    res[scols].to_csv('submission.csv', index=False)

In [32]:
if not train:  # Only run during inference, not training
    subs = pd.read_csv('submission.csv')
    subs